# Wheel-speed variance at several speeds

Measures, in the same runs:
1. the **conversion factor** Thymio units → mm/s, precisely, from the time the ground sensors take to go from one floor mark to the other (1.50 m apart);
2. the **variance of each wheel's speed** at targets 100, 150 and 200.

**Before you start**
- Thymio on, Thymio Suite open, Aseba Studio closed.
- Two marks across the lane, `MARK_DISTANCE_MM` apart (measure centre-to-centre or edge-to-edge, the same edge on both). They must contrast with the floor (e.g. black tape on a light floor) and be at least 2 cm wide.
- Start every run with the robot's **front** `START_OFFSET_MM` before the first mark, pointing along the lane, so it is already at constant speed when it reaches the mark.
- Between runs the notebook stops, asks you to put the robot back, and waits for Enter.

Run the cells in order. Raw data is saved to `wheel_variance_data.json` after every run, so nothing is lost if something crashes.

In [1]:
import time, json, math
import numpy as np
import matplotlib.pyplot as plt
from tdmclient import ClientAsync

client = ClientAsync()
node = await client.wait_for_node()
await node.lock()
print("Connected and locked")

Connected and locked


## Settings

In [ ]:
SPEEDS = [100, 150, 200]     # motor targets to test (Thymio units)
TOTAL_TIME = 40              # seconds of steady driving to collect per speed (summed over runs)
MAX_RUN_TIME = 70            # safety cap on a single run (s)
TS = 0.1                     # sampling period (s)
SKIP = 1.5                   # seconds dropped at the start of each run (acceleration)

MARK_DISTANCE_MM = 500      # distance between the two floor marks (same edge to same edge)
START_OFFSET_MM = 0        # robot front placed this far before the first mark
OVERSHOOT_MM = 100           # keep driving this far after the second mark, then stop
MARK_IS_DARK = True          # True: dark tape on light floor. False: light tape on dark floor
RUN_DISTANCE_MM = START_OFFSET_MM + MARK_DISTANCE_MM + OVERSHOOT_MM

FACTOR_L = 0.380             # mm/s per Thymio unit, left wheel (your calibration)
FACTOR_R = 0.370             # mm/s per Thymio unit, right wheel
FACTOR_AVG = (FACTOR_L + FACTOR_R) / 2

DATA_FILE = "wheel_variance_data.json"

def to_signed(v):
    return v - 65536 if v > 32767 else v

def motors(left, right):
    return {"motor.left.target": [left], "motor.right.target": [right]}

# plan: run length and number of runs for each speed
plan = {}
for s in SPEEDS:
    expected_mms = s * FACTOR_AVG
    run_time = min(MAX_RUN_TIME, RUN_DISTANCE_MM / expected_mms)
    n_runs = max(2, math.ceil(TOTAL_TIME / (run_time - SKIP)))
    plan[s] = (round(run_time, 1), n_runs)
    print(f"target {s}: ~{expected_mms:.0f} mm/s -> {n_runs} runs of {run_time:.1f} s")

target 100: ~38 mm/s -> 2 runs of 45.3 s
target 150: ~56 mm/s -> 2 runs of 30.2 s
target 200: ~75 mm/s -> 2 runs of 22.7 s


## Recording function

In [3]:
async def record_run(target, duration):
    """Drive straight at `target` for `duration` s; record both wheel speeds and both ground sensors."""
    await node.wait_for_variables({"motor.left.speed", "motor.right.speed", "prox.ground.reflected"})
    t, vl, vr, gl, gr = [], [], [], [], []
    try:
        await node.set_variables(motors(target, target))
        t0 = time.time()                          # clock starts once the motors are on
        while time.time() - t0 < duration:
            t.append(time.time() - t0)
            vl.append(to_signed(node["motor.left.speed"]))
            vr.append(to_signed(node["motor.right.speed"]))
            ground = list(node["prox.ground.reflected"])
            gl.append(ground[0]); gr.append(ground[1])
            await client.sleep(TS)                # not time.sleep
    finally:
        await node.set_variables(motors(0, 0))    # always stop
    return t, vl, vr, gl, gr

## Run the experiment

Follow the prompts. Type `s` then Enter at a prompt to skip the remaining runs of that speed.

In [4]:
runs = []   # list of dicts: {"target", "run", "t", "vl", "vr"}

for target in SPEEDS:
    run_time, n_runs = plan[target]
    for r in range(n_runs):
        answer = input(f"Target {target}, run {r+1}/{n_runs} ({run_time} s). Place the robot at the start, then press Enter (s = skip): ")
        if answer.strip().lower() == "s":
            break
        t, vl, vr, gl, gr = await record_run(target, run_time)
        runs.append({"target": target, "run": r + 1, "t": t, "vl": vl, "vr": vr, "gl": gl, "gr": gr})
        with open(DATA_FILE, "w") as fh:
            json.dump(runs, fh)
        dt = np.diff(t)
        print(f"  {len(t)} samples, mean dt {dt.mean():.3f} s, max dt {dt.max():.3f} s")

print("Done. Data saved to", DATA_FILE)

  438 samples, mean dt 0.103 s, max dt 0.170 s


CancelledError: 

## Check the raw data

Each run should be flat (on average) after the dashed line. If a curve still ramps up after it, increase `SKIP`. A run with a bump (robot hit something, was touched) can be removed from `runs` before the analysis.

In [ ]:
# to reload saved data in a new session:  runs = json.load(open(DATA_FILE))

fig, axes = plt.subplots(1, len(SPEEDS), figsize=(5 * len(SPEEDS), 4), sharey=True)
for ax, target in zip(np.atleast_1d(axes), SPEEDS):
    for run in [r for r in runs if r["target"] == target]:
        ax.plot(run["t"], run["vl"], color="C0", alpha=0.6)
        ax.plot(run["t"], run["vr"], color="C1", alpha=0.6)
    ax.axvline(SKIP, color="gray", linestyle="--")
    ax.axhline(target, color="black", linestyle=":")
    ax.set_title(f"target {target} (blue = left, orange = right)")
    ax.set_xlabel("time (s)")
np.atleast_1d(axes)[0].set_ylabel("measured speed (Thymio units)")
plt.show()

## Conversion factor from the floor marks

For each run and each ground sensor, the notebook finds the moment the sensor enters each mark (threshold halfway between floor and tape, interpolated between samples for sub-sample precision). The two sensors' times are averaged, which cancels a slightly angled robot.

Real speed = `MARK_DISTANCE_MM` / (t₂ − t₁). It is compared with the mean measured wheel speeds over the same interval. The ruler measurement fixed the **ratio** between the left and right factors; the marks fix their **scale** precisely.

In [ ]:
def mark_entries(t, g, dark=MARK_IS_DARK, debounce=0.5):
    """Times at which the sensor enters a mark, interpolated between samples."""
    t = np.asarray(t, float); g = np.asarray(g, float)
    floor = np.median(g)
    tape = g.min() if dark else g.max()
    thr = (floor + tape) / 2
    on = g < thr if dark else g > thr
    times = []
    for k in range(1, len(g)):
        if on[k] and not on[k - 1]:
            frac = (thr - g[k - 1]) / (g[k] - g[k - 1])
            tc = t[k - 1] + frac * (t[k] - t[k - 1])
            if not times or tc - times[-1] > debounce:
                times.append(tc)
    return times, thr

ratio_l, ratio_r = FACTOR_L, FACTOR_R      # ruler values: only their ratio is kept
factor_rows = []
for run in runs:
    t = np.array(run["t"])
    el, thr_l = mark_entries(t, run["gl"])
    er, thr_r = mark_entries(t, run["gr"])
    run["entries"] = (el, er, thr_l, thr_r)
    if len(el) != 2 or len(er) != 2:
        print(f"target {run['target']} run {run['run']}: found {len(el)} / {len(er)} mark entries (need 2 / 2), skipped")
        continue
    t1 = (el[0] + er[0]) / 2
    t2 = (el[1] + er[1]) / 2
    if t1 < SKIP:
        print(f"target {run['target']} run {run['run']}: first mark reached at {t1:.2f} s, before SKIP; increase START_OFFSET_MM")
    v_real = MARK_DISTANCE_MM / (t2 - t1)                       # mm/s, robot centre
    inside = (t >= t1) & (t <= t2)
    vl_mean = np.mean(np.array(run["vl"])[inside])
    vr_mean = np.mean(np.array(run["vr"])[inside])
    scale = 2 * v_real / (ratio_l * vl_mean + ratio_r * vr_mean)
    factor_rows.append({"target": run["target"], "run": run["run"], "v_real": v_real,
                        "dt": t2 - t1, "f_l": scale * ratio_l, "f_r": scale * ratio_r,
                        "f_avg": v_real / ((vl_mean + vr_mean) / 2)})

print(f"{'target':>6} {'run':>3} {'t2-t1 (s)':>9} {'v real (mm/s)':>13} {'factor avg':>10} {'factor L':>9} {'factor R':>9}")
for r in factor_rows:
    print(f"{r['target']:>6} {r['run']:>3} {r['dt']:>9.2f} {r['v_real']:>13.2f} {r['f_avg']:>10.4f} {r['f_l']:>9.4f} {r['f_r']:>9.4f}")

if factor_rows:
    f_avg = np.array([r["f_avg"] for r in factor_rows])
    print(f"\nAverage factor: {f_avg.mean():.4f} mm/s per unit (spread between runs: {f_avg.std():.4f})")
    FACTOR_L = float(np.mean([r["f_l"] for r in factor_rows]))
    FACTOR_R = float(np.mean([r["f_r"] for r in factor_rows]))
    print(f"Updated FACTOR_L = {FACTOR_L:.4f}, FACTOR_R = {FACTOR_R:.4f}  (used in the variance analysis below)")

Check that the detected entries (crosses) sit at the start of each dip. If a run shows extra or missing entries, adjust `MARK_IS_DARK`, the tape, or remove that run.

In [ ]:
n = len(runs)
fig, axes = plt.subplots(n, 1, figsize=(10, 2.2 * n), sharex=False)
for ax, run in zip(np.atleast_1d(axes), runs):
    el, er, thr_l, thr_r = run["entries"]
    ax.plot(run["t"], run["gl"], color="C0", label="left ground")
    ax.plot(run["t"], run["gr"], color="C1", label="right ground")
    ax.axhline(thr_l, color="C0", linestyle=":"); ax.axhline(thr_r, color="C1", linestyle=":")
    for tc in el: ax.plot(tc, thr_l, "x", color="C0", markersize=10, mew=2)
    for tc in er: ax.plot(tc, thr_r, "x", color="C1", markersize=10, mew=2)
    ax.set_title(f"target {run['target']} run {run['run']}", fontsize=10)
np.atleast_1d(axes)[0].legend(loc="upper right")
np.atleast_1d(axes)[-1].set_xlabel("time (s)")
plt.tight_layout(); plt.show()

## Variance per run and per speed

Each run's variance is computed around that run's own mean. The per-speed value pools the runs (weighted by their number of samples), so small mean differences between runs don't inflate it.

In [ ]:
rows = []
for run in runs:
    t = np.array(run["t"]); keep = t > SKIP
    vl = np.array(run["vl"])[keep]; vr = np.array(run["vr"])[keep]
    rows.append({"target": run["target"], "run": run["run"], "n": keep.sum(),
                 "mean_l": vl.mean(), "mean_r": vr.mean(),
                 "var_l": vl.var() * FACTOR_L**2, "var_r": vr.var() * FACTOR_R**2})   # (mm/s)^2

print(f"{'target':>6} {'run':>3} {'n':>4} {'mean L':>7} {'mean R':>7} {'var L (mm/s)^2':>15} {'var R (mm/s)^2':>15}")
for r in rows:
    print(f"{r['target']:>6} {r['run']:>3} {r['n']:>4} {r['mean_l']:>7.1f} {r['mean_r']:>7.1f} {r['var_l']:>15.2f} {r['var_r']:>15.2f}")

summary = {}
for target in SPEEDS:
    rs = [r for r in rows if r["target"] == target]
    if not rs:
        continue
    n = np.array([r["n"] for r in rs])
    summary[target] = {
        "speed_l_mms": np.average([r["mean_l"] for r in rs], weights=n) * FACTOR_L,
        "speed_r_mms": np.average([r["mean_r"] for r in rs], weights=n) * FACTOR_R,
        "var_l": np.average([r["var_l"] for r in rs], weights=n),
        "var_r": np.average([r["var_r"] for r in rs], weights=n),
    }

print("\nPooled per speed:")
for target, s in summary.items():
    print(f"target {target}: left {s['speed_l_mms']:.1f} mm/s, var {s['var_l']:.2f} | "
          f"right {s['speed_r_mms']:.1f} mm/s, var {s['var_r']:.2f}  (mm/s)^2")

## Does the noise depend on speed?

Dots = single runs, line = pooled value per speed. If the points are roughly flat, use one constant M. If they grow, either take the largest value or use the linear fit below.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for r in rows:
    ax.plot(r["mean_l"] * FACTOR_L, r["var_l"], "o", color="C0", alpha=0.5)
    ax.plot(r["mean_r"] * FACTOR_R, r["var_r"], "o", color="C1", alpha=0.5)
sp_l = [s["speed_l_mms"] for s in summary.values()]; vv_l = [s["var_l"] for s in summary.values()]
sp_r = [s["speed_r_mms"] for s in summary.values()]; vv_r = [s["var_r"] for s in summary.values()]
ax.plot(sp_l, vv_l, "-", color="C0", label="left")
ax.plot(sp_r, vv_r, "-", color="C1", label="right")
ax.set_xlabel("wheel speed (mm/s)"); ax.set_ylabel("variance (mm/s)^2")
ax.set_ylim(bottom=0); ax.legend(); plt.show()

# Option 1: one constant M, the safe choice (largest pooled value)
var_max = max(vv_l + vv_r)
M_const = np.diag([var_max, var_max])
print(f"Option 1, constant M = diag({var_max:.2f}, {var_max:.2f})  (mm/s)^2")

# Option 2: linear fit  var = a + b*|v|, one per wheel, if you have >= 2 speeds
if len(summary) >= 2:
    b_l, a_l = np.polyfit(sp_l, vv_l, 1)
    b_r, a_r = np.polyfit(sp_r, vv_r, 1)
    print(f"Option 2, left : var = {a_l:.2f} + {b_l:.4f} * |v_l|")
    print(f"          right: var = {a_r:.2f} + {b_r:.4f} * |v_r|")
    print("          use in your loop:  M = np.diag([a_r + b_r*abs(v_r), a_l + b_l*abs(v_l)])  (order v_r, v_l)")

## Release the robot

In [14]:
await node.set_variables(motors(0, 0))
await node.unlock()
print("Robot released")

Robot released
